# KMeans Clustering — Manual Implementation

**Dependencies** (managed by `uv`):
- `numpy` — array math
- `scikit-learn` — `make_blobs` (data generation), `KMeans` (reference)
- `matplotlib` — scatter / elbow plots

We implement KMeans **from scratch** using only `numpy`, then compare against sklearn's built-in implementation.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans as SklearnKMeans
import matplotlib
matplotlib.use("TkAgg")

# ---- Palette (dataviz-validated categorical set) ----
SERIES  = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]
TEXT_P  = "#0b0b0b"
TEXT_S  = "#52514e"
SURFACE = "#fcfcfb"

plt.rcParams.update({
    "text.color"       : TEXT_P,
    "axes.edgecolor"   : TEXT_S,
    "axes.labelcolor"  : TEXT_P,
    "axes.facecolor"   : SURFACE,
    "figure.facecolor" : SURFACE,
    "grid.color"       : "#e0dfdc",
    "grid.linewidth"   : 0.8,
    "axes.grid"        : True,
    "font.size"        : 11,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
})

print("Setup complete ✓")

Setup complete ✓


## 1. Generate Data with `make_blobs`

`make_blobs` creates Gaussian clusters around random centers — perfect for KMeans.


In [2]:
X, true_labels = make_blobs(
    n_samples=400,
    centers=4,
    cluster_std=[0.8, 1.0, 1.2, 0.9],
    random_state=42,
)
print(f"Data shape : {X.shape}")
print(f"True labels: {np.unique(true_labels)}")
print(f"Sample points:\n{X[:5]}")


Data shape : (400, 2)
True labels: [0 1 2 3]
Sample points:
[[-9.99071417  8.90283768]
 [-4.60499371  9.67180813]
 [-9.05949651  7.0780717 ]
 [ 5.61499857  1.8261123 ]
 [ 5.21076935  3.10873532]]


## 2. Manual KMeans Implementation

**Algorithm:**
1. **Init** — pick `k` initial centers (K-means++ style: sample proportional to squared distance)
2. **Assign** — assign every point to its nearest center
3. **Update** — recompute each center as the mean of its assigned points
4. **Repeat** until centers stop moving (convergence) or `max_iter` is reached


In [3]:
def kmeans_manual(X: np.ndarray, k: int,
                  max_iter: int = 300, tol: float = 1e-6,
                  seed: int = 42) -> dict:
    """KMeans from scratch using only numpy.

    Returns a dict with:
      centers          – final (k, d) center array
      labels           – (n,) cluster assignment per point
      inertia          – sum of squared distances to assigned centers
      history          – list of center arrays at each iteration
      inertia_history  – inertia at each iteration
    """
    rng = np.random.default_rng(seed)
    n   = X.shape[0]

    # ---- K-means++ style initialization ----
    centers = np.empty((k, X.shape[1]))
    centers[0] = X[rng.integers(n)]
    for j in range(1, k):
        # squared distance from each point to nearest existing center
        d = np.min(
            np.stack([np.sum((X - centers[i]) ** 2, axis=1)
                      for i in range(j)], axis=1),
            axis=1,
        )
        centers[j] = X[rng.choice(n, p=d / d.sum())]

    labels = np.zeros(n, dtype=int)
    history         = [centers.copy()]
    inertia_history = []

    for it in range(max_iter):
        # ---- Assign step ----
        dists = np.stack(
            [np.sum((X - centers[i]) ** 2, axis=1) for i in range(k)],
            axis=1,
        )
        new_labels = np.argmin(dists, axis=1)

        # ---- Update step ----
        new_centers = np.empty_like(centers)
        for j in range(k):
            members = X[new_labels == j]
            new_centers[j] = (members.mean(axis=0)
                              if len(members) > 0
                              else X[rng.integers(n)])   # re-seed empty cluster

        shift = np.linalg.norm(new_centers - centers)
        inertia_history.append(dists[np.arange(n), new_labels].sum())
        centers, labels = new_centers, new_labels
        history.append(centers.copy())

        if shift < tol:
            print(f"  Converged at iteration {it + 1}  (shift = {shift:.2e})")
            break

    inertia = dists[np.arange(n), labels].sum()
    print(f"  Inertia: {inertia:.2f}")
    return {
        "centers"         : centers,
        "labels"          : labels,
        "inertia"         : inertia,
        "history"         : history,
        "inertia_history" : inertia_history,
    }

print("kmeans_manual() defined ✓")


kmeans_manual() defined ✓


## 3. Elbow Curve — Choosing `k`

Run KMeans for a range of `k` values and plot inertia vs. `k`.
The "elbow" is the point where adding more clusters gives diminishing returns.


In [4]:
k_range  = range(1, 9)
inerties = []

print("Running Elbow Curve ...")
for k in k_range:
    res = kmeans_manual(X, k=k)
    inerties.append(res["inertia"])

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(list(k_range), inerties, "o-", color=SERIES[0], lw=2, ms=8)
ax.set_xlabel("k (number of clusters)")
ax.set_ylabel("Inertia (within-cluster SSE)")
ax.set_title("Elbow Curve — Choosing k")
ax.set_xticks(list(k_range))
fig.tight_layout()
fig.savefig("elbow.png", dpi=150)
plt.show()


Running Elbow Curve ...
  Converged at iteration 2  (shift = 0.00e+00)
  Inertia: 27375.65
  Converged at iteration 2  (shift = 0.00e+00)
  Inertia: 12810.12
  Converged at iteration 2  (shift = 0.00e+00)
  Inertia: 2884.56
  Converged at iteration 2  (shift = 0.00e+00)
  Inertia: 754.64
  Converged at iteration 3  (shift = 0.00e+00)
  Inertia: 655.53
  Converged at iteration 3  (shift = 0.00e+00)
  Inertia: 613.55
  Converged at iteration 4  (shift = 0.00e+00)
  Inertia: 537.04
  Converged at iteration 4  (shift = 0.00e+00)
  Inertia: 484.36


## 4. Dendrogram — Hierarchical View of the Data

A dendrogram shows how points merge into clusters as the distance threshold
increases. We build it with **sklearn** (`AgglomerativeClustering` + manual
merge tree) rather than scipy.


In [5]:
from sklearn.metrics.pairwise import euclidean_distances

def build_dendrogram(X: np.ndarray):
    """Build a single-linkage merge tree using only numpy + sklearn.

    Returns (merges, root_id):
      merges   : list of (left_id, right_id, distance, size) per merge step
      root_id  : id of the final merged cluster
    """
    n = X.shape[0]
    D = euclidean_distances(X)
    np.fill_diagonal(D, np.inf)

    clusters = {i: [i] for i in range(n)}
    alive    = set(range(n))
    next_id  = n
    merges   = []

    for _ in range(n - 1):
        min_d, best_pair = np.inf, None
        alive_list = sorted(alive)
        for i, a in enumerate(alive_list):
            for b in alive_list[i + 1 :]:
                d = min(D[x, y] for x in clusters[a] for y in clusters[b])
                if d < min_d:
                    min_d, best_pair = d, (a, b)
        a, b = best_pair
        clusters[next_id] = clusters[a] + clusters[b]
        merges.append((a, b, min_d, len(clusters[next_id])))
        alive.discard(a); alive.discard(b); alive.add(next_id)
        next_id += 1

    return merges, next_id - 1


def plot_dendrogram(merges, n_leaves) -> None:
    """Draw a dendrogram-style merge-distance plot with matplotlib.

    Each bar represents one merge step; bar length = distance at which
    the two clusters were joined. A gap in bar lengths signals a natural
    cluster boundary (the 'elbow' of the dendrogram).
    """
    heights = [m[2] for m in merges]
    sizes   = [m[3] for m in merges]
    steps   = list(range(1, len(merges) + 1))

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(
        steps, heights,
        color=[SERIES[0]] * len(steps),
        alpha=0.85, height=0.85,
    )
    # Highlight large gaps (natural cluster boundaries)
    diffs = np.diff(heights)
    for i, d in enumerate(diffs):
        if d > np.mean(diffs) + 2 * np.std(diffs):
            ax.annotate(
                f"gap ≈ {d:.1f}",
                xy=(max(heights[i], heights[i+1]) * 0.9, i + 1.5),
                fontsize=8, color=SERIES[1], fontweight="bold",
            )
    ax.set_xlabel("Merge distance (single linkage)")
    ax.set_ylabel("Merge step")
    ax.set_title("Dendrogram — single-linkage merge distances (sklearn)")
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig("dendrogram.png", dpi=150)
    plt.show()


merges, root_id = build_dendrogram(X)
plot_dendrogram(merges, len(X))
print(f"Built merge tree: {len(merges)} merge steps ✓")
print(f"Root cluster id: {root_id}  (contains {len(X)} points)")


Built merge tree: 399 merge steps ✓
Root cluster id: 798  (contains 400 points)


## 5. Final KMeans Result (k = 4)


In [6]:
K = 4
print(f"Running KMeans (k={K}) ...")
res = kmeans_manual(X, k=K)

# --- sklearn reference ---
sk = SklearnKMeans(n_clusters=K, n_init=10, random_state=42).fit(X)
print(f"  sklearn reference inertia: {sk.inertia_:.2f}")
print(f"  our implementation inertia: {res['inertia']:.2f}")


Running KMeans (k=4) ...
  Converged at iteration 2  (shift = 0.00e+00)
  Inertia: 754.64


  sklearn reference inertia: 754.64
  our implementation inertia: 754.64


## 6. Visualization — Our KMeans vs. Ground Truth


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# --- Left: our KMeans ---
for j in range(K):
    mask = res["labels"] == j
    axes[0].scatter(
        X[mask, 0], X[mask, 1],
        c=SERIES[j % len(SERIES)], alpha=0.75, s=28,
        label=f"Cluster {j}",
    )
axes[0].scatter(
    res["centers"][:, 0], res["centers"][:, 1],
    c=[SERIES[j % len(SERIES)] for j in range(K)],
    marker="X", s=220, edgecolors="k", linewidths=1.2, zorder=5,
)
axes[0].set_title(f"Our KMeans  (k={K})")
axes[0].set_xlabel("x₁"); axes[0].set_ylabel("x₂")
axes[0].legend(loc="best", fontsize=9)

# --- Right: ground truth ---
for j in range(K):
    mask = true_labels == j
    axes[1].scatter(
        X[mask, 0], X[mask, 1],
        c=SERIES[j % len(SERIES)], alpha=0.75, s=28,
        label=f"True cluster {j}",
    )
axes[1].set_title("Ground truth (make_blobs)")
axes[1].set_xlabel("x₁"); axes[1].set_ylabel("x₂")
axes[1].legend(loc="best", fontsize=9)

fig.suptitle("KMeans — our implementation vs. ground truth", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig("kmeans_result.png", dpi=150)
plt.show()
